In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/hashirali24/pakistan-house-prices/Cleaned_data_for_model.csv


In [2]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/hashirali24/pakistan-house-prices/Cleaned_data_for_model.csv


In [3]:
import pandas as pd

path = "/kaggle/input/datasets/hashirali24/pakistan-house-prices/Cleaned_data_for_model.csv"
df = pd.read_csv(path)

# Sirf bikri wale ghar lein (rent wale alag hain)
df = df[df["purpose"] == "For Sale"].drop(columns=["Unnamed: 0", "purpose"])

print(df.shape)
print(df.head())
print(df.describe())

(70947, 7)
  property_type     price     location       city  baths  bedrooms  \
0          Flat  10000000         G-10  Islamabad      2         2   
1          Flat   6900000         E-11  Islamabad      3         3   
2         House  16500000         G-15  Islamabad      6         5   
3         House  43500000    Bani Gala  Islamabad      4         4   
4         House   7000000  DHA Defence  Islamabad      3         3   

   Area_in_Marla  
0            4.0  
1            5.6  
2            8.0  
3           40.0  
4            8.0  
              price         baths      bedrooms  Area_in_Marla
count  7.094700e+04  70947.000000  70947.000000   70947.000000
mean   1.451733e+07      3.635982      3.445798       7.447420
std    9.979440e+06      1.504327      1.258346       5.893361
min    1.600000e+04      1.000000      0.000000       0.000000
25%    6.700000e+06      2.000000      3.000000       4.400000
50%    1.200000e+07      3.000000      3.000000       5.800000
75%    2.0000

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import joblib

# Data load
path = "/kaggle/input/datasets/hashirali24/pakistan-house-prices/Cleaned_data_for_model.csv"
df = pd.read_csv(path)
df = df[df["purpose"] == "For Sale"].drop(columns=["Unnamed: 0", "purpose"])

# 1) Galat rows hatao
d = df[(df.price >= 500000) & (df.Area_in_Marla >= 1) &
       (df.Area_in_Marla <= 100) & (df.bedrooms >= 1)].copy()
print("Rows after cleaning:", d.shape[0])

# 2) Features aur target
cat = ["property_type", "location", "city"]
num = ["baths", "bedrooms", "Area_in_Marla"]
X = d[cat + num]
y = np.log1p(d["price"])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

# 3) Model
pre = ColumnTransformer(
    [("c", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat)],
    remainder="passthrough")
model = Pipeline([("pre", pre), ("gb", HistGradientBoostingRegressor(max_iter=300, random_state=42))])
model.fit(Xtr, ytr)

# 4) Result
pred = np.expm1(model.predict(Xte))
actual = np.expm1(yte)
print("R2 score:", round(r2_score(actual, pred), 3))
print("Average error (PKR):", int(mean_absolute_error(actual, pred)))

# 5) Model save
joblib.dump(model, "/kaggle/working/house_price_model.pkl")

Rows after cleaning: 70630
R2 score: 0.855
Average error (PKR): 2439036


['/kaggle/working/house_price_model.pkl']